# Phase 07J: Android Integration Compatibility Preflight

Phase 07J kiểm tra bundle ONNX Runtime INT8 đã khóa ở Phase 07I so với
runtime contract của ứng dụng Android V1 hiện tại. Notebook **không thay
trực tiếp mã nguồn Android** và không benchmark thiết bị vật lý.

Mục tiêu:

- xác minh toàn vẹn deployment bundle của ba cặp;
- chỉ ra các thay đổi bắt buộc khi chuyển V1 FP32 sang V2 INT8;
- cấm thay asset trực tiếp nếu tensor/vocabulary contract không tương thích;
- tạo Android integration package và model registry bất biến;
- giữ V1 demo làm fallback;
- cấp quyền Phase 07K thực hiện source migration và build APK.

## Kaggle inputs

1. Output Phase 07I ORT Quality Aggregate Deployment Lock.

## Runtime

- Accelerator: `None / CPU`.
- Internet: `Off`.
- Không cần ảnh MS COCO, không huấn luyện và không gắn Held-out Test.

## Cell 1 — Phase 07I authorization và manifest integrity

In [1]:
import csv
import hashlib
import json
import os
import platform
import shutil
import time
from datetime import datetime, timezone
from pathlib import Path

STARTED = time.perf_counter()
INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))
NOTEBOOK_ROOT = (
    INPUT_ROOT / "notebooks"
    if (INPUT_ROOT / "notebooks").is_dir()
    else INPUT_ROOT
)
OUT = Path(os.environ.get(
    "KLCN_OUTPUT_ROOT",
    "/kaggle/working//phase7j_android_integration_preflight",
))
AUDIT = OUT / "audits"
CONFIG = OUT / "configs"
RESULT = OUT / "results"
PACKAGE = OUT / "android_integration_package"
MANIFEST = OUT / "manifests"
for directory in [OUT, AUDIT, CONFIG, RESULT, PACKAGE, MANIFEST]:
    directory.mkdir(parents=True, exist_ok=True)

def now():
    return datetime.now(timezone.utc).isoformat()

def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)

def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    temporary.replace(path)

def sha256(path, block_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            block = handle.read(block_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()

def exactly_one(name):
    matches = sorted(NOTEBOOK_ROOT.rglob(name))
    assert len(matches) == 1, (
        f"Cần đúng một {name}; tìm thấy {len(matches)}: {matches}"
    )
    return matches[0].resolve()

def resolve_inside(root, relative_path):
    root = Path(root).resolve()
    target = (root / relative_path).resolve()
    assert target == root or root in target.parents
    return target

P7I_PATH = exactly_one(
    "PHASE7I_ORT_QUALITY_AGGREGATE_DEPLOYMENT_LOCK_COMPLETE.json"
)
P7J_READY_PATH = exactly_one("PHASE7J_ANDROID_INTEGRATION_INPUT_READY.json")
P7I = read_json(P7I_PATH)
P7J_READY = read_json(P7J_READY_PATH)
P7I_ROOT = P7I_PATH.parent

assert P7I["status"] == "complete"
assert P7I["decision"] == (
    "three_ort_int8_candidates_locked_phase7j_authorized"
)
assert P7I["pairs_verified"] == P7I["pairs_quality_passed"] == 3
assert P7I["validation_images_per_pair"] == 5000
assert P7I["deployment_bundle_complete"] is True
assert P7I["android_application_integration_authorized"] is True
assert P7I["physical_android_benchmark_performed"] is False
assert P7I["new_training_performed"] is False
assert P7I["checkpoint_reselection_performed"] is False
assert P7I["test_set_status"] == "sealed" and P7I["test_set_used"] is False

assert P7J_READY["status"] == "ready"
assert P7J_READY["runtime"] == "onnxruntime-android:1.29.0"
assert P7J_READY["pairs"] == ["T1-S1", "T2-S2", "T3-S3"]
assert P7J_READY["android_application_integration_authorized"] is True
assert P7J_READY["physical_android_benchmark_authorized"] is False
assert P7J_READY["new_training"] is False
assert P7J_READY["checkpoint_reselection"] is False
assert P7J_READY["test_set_status"] == "sealed"
assert P7J_READY["test_set_used"] is False

manifest_ref = P7I["manifest"]["csv"]
manifest_path = resolve_inside(P7I_ROOT, manifest_ref["relative_path"])
assert manifest_path.is_file()
assert manifest_path.stat().st_size == int(manifest_ref["size_bytes"])
assert sha256(manifest_path) == manifest_ref["sha256"]
with manifest_path.open("r", encoding="utf-8", newline="") as handle:
    p7i_manifest_rows = list(csv.DictReader(handle))
assert len(p7i_manifest_rows) == int(manifest_ref["rows"]) == 29
for row in p7i_manifest_rows:
    artifact = resolve_inside(P7I_ROOT, row["relative_path"])
    assert artifact.is_file()
    assert artifact.stat().st_size == int(row["size_bytes"])
    assert sha256(artifact) == row["sha256"]

for role in ["deployment_lock", "bundle_registry"]:
    reference = P7J_READY[role]
    artifact = resolve_inside(P7I_ROOT, reference["relative_path"])
    assert artifact.stat().st_size == int(reference["size_bytes"])
    assert sha256(artifact) == reference["sha256"]

DEPLOYMENT_LOCK_PATH = resolve_inside(
    P7I_ROOT, P7J_READY["deployment_lock"]["relative_path"]
)
BUNDLE_REGISTRY_PATH = resolve_inside(
    P7I_ROOT, P7J_READY["bundle_registry"]["relative_path"]
)
DEPLOYMENT_LOCK = read_json(DEPLOYMENT_LOCK_PATH)
BUNDLE_REGISTRY = read_json(BUNDLE_REGISTRY_PATH)
assert DEPLOYMENT_LOCK["status"] == "locked"
assert DEPLOYMENT_LOCK["android_application_integration_authorized"] is True
assert DEPLOYMENT_LOCK["physical_android_benchmark_authorized"] is False
assert BUNDLE_REGISTRY["runtime"] == "onnxruntime-android:1.29.0"
assert BUNDLE_REGISTRY["vocabulary_size"] == 10080
assert len(BUNDLE_REGISTRY["pairs"]) == 3

write_json(AUDIT / "phase7j_cell01_entry.json", {
    "schema_version": "phase7j_cell01_entry_v1",
    "created_at_utc": now(),
    "status": "passed",
    "phase7i_manifest_rows": len(p7i_manifest_rows),
    "pairs_authorized": P7J_READY["pairs"],
    "runtime": P7J_READY["runtime"],
    "android_application_integration_authorized": True,
    "physical_android_benchmark_authorized": False,
    "new_training_started": False,
    "checkpoint_reselection": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})

print("PHASE 07J CELL 1 — ENTRY")
print("Status                  : PASSED")
print("Phase 07I manifest      : 29 / 29 VERIFIED")
print("Deployment lock         : VERIFIED")
print("Pair bundles            : 3 / 3 AUTHORIZED")
print("Runtime                 : onnxruntime-android 1.29.0")
print("Physical benchmark      : NOT AUTHORIZED")
print("Held-out Test           : SEALED / NOT USED")
print("Progress                : 1 / 5 code cells")

PHASE 07J CELL 1 — ENTRY
Status                  : PASSED
Phase 07I manifest      : 29 / 29 VERIFIED
Deployment lock         : VERIFIED
Pair bundles            : 3 / 3 AUTHORIZED
Runtime                 : onnxruntime-android 1.29.0
Physical benchmark      : NOT AUTHORIZED
Held-out Test           : SEALED / NOT USED
Progress                : 1 / 5 code cells


## Cell 2 — Bundle contract và artifact audit

In [2]:
STARTED = time.perf_counter()
BUNDLE_ROOT = P7I_ROOT / "deployment_bundle"
assert BUNDLE_ROOT.is_dir()

PAIR_RUNTIME = {}
artifact_rows = []
for pair_entry in BUNDLE_REGISTRY["pairs"]:
    pair = pair_entry["pair"]
    runtime_ref = pair_entry["runtime_contract"]
    runtime_path = resolve_inside(BUNDLE_ROOT, runtime_ref["relative_path"])
    assert runtime_path.stat().st_size == int(runtime_ref["size_bytes"])
    assert sha256(runtime_path) == runtime_ref["sha256"]
    contract = read_json(runtime_path)
    assert contract["pair"] == pair
    assert contract["runtime"] == {
        "name": "onnxruntime-android",
        "version": "1.29.0",
        "opset": 17,
        "cpu_threads_for_validation": 1,
    }
    assert contract["tensor_contract"] == {
        "image": [1, 3, 224, 224],
        "visual_memory": [1, 49, 256],
        "decoder_input_ids": [1, 31],
        "decoder_attention_mask": [1, 31],
        "logits": [1, 31, 10080],
    }
    assert contract["tensor_names"] == {
        "encoder_inputs": ["image"],
        "encoder_outputs": ["visual_memory"],
        "decoder_inputs": [
            "decoder_input_ids",
            "visual_memory",
            "decoder_attention_mask",
        ],
        "decoder_outputs": ["logits"],
    }
    assert contract["special_token_ids"] == {
        "pad": 0, "bos": 1, "eos": 2, "unk": 3,
    }
    assert contract["decoding"] == {
        "beam_size": 3,
        "length_penalty": 0.7,
        "maximum_steps": 31,
    }
    assert contract["validation_quality"]["quality_gate_passed"] is True
    assert contract["physical_android_benchmark_performed"] is False
    assert contract["test_set_status"] == "sealed"
    assert contract["test_set_used"] is False

    for artifact_name, reference in pair_entry["artifacts"].items():
        artifact_path = resolve_inside(BUNDLE_ROOT, reference["relative_path"])
        assert artifact_path.is_file()
        assert artifact_path.stat().st_size == int(reference["size_bytes"])
        assert sha256(artifact_path) == reference["sha256"]
        artifact_rows.append({
            "pair": pair,
            "artifact": artifact_name,
            "relative_path": reference["relative_path"],
            "size_bytes": int(reference["size_bytes"]),
            "sha256": reference["sha256"],
        })
    PAIR_RUNTIME[pair] = {
        "registry": pair_entry,
        "contract": contract,
        "runtime_path": runtime_path,
    }

assert len(PAIR_RUNTIME) == 3
assert len(artifact_rows) == 6
shared_files = [
    "shared/word_vocabulary_minfreq5.json",
    "shared/tokenizer_contract.json",
    "shared/sequence_encoding_contract.json",
]
for relative_path in shared_files:
    assert (BUNDLE_ROOT / relative_path).is_file()
vocabulary = read_json(BUNDLE_ROOT / shared_files[0])
assert len(vocabulary["id_to_token"]) == 10080
assert vocabulary["id_to_token"][:4] == [
    "<pad>", "<bos>", "<eos>", "<unk>",
]

artifact_csv = RESULT / "phase7j_android_artifact_inventory.csv"
with artifact_csv.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(artifact_rows[0]))
    writer.writeheader()
    writer.writerows(artifact_rows)
write_json(AUDIT / "phase7j_cell02_bundle_audit.json", {
    "schema_version": "phase7j_cell02_bundle_audit_v1",
    "created_at_utc": now(),
    "status": "passed",
    "pairs_verified": 3,
    "onnx_artifacts_verified": 6,
    "vocabulary_size": 10080,
    "tensor_contract_verified": True,
    "physical_android_benchmark_performed": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})

print("PHASE 07J CELL 2 — BUNDLE AUDIT")
print("Status                  : PASSED")
print("Runtime contracts       : 3 / 3 VERIFIED")
print("ONNX artifacts          : 6 / 6 VERIFIED")
print("Vocabulary              : 10,080")
print("Tensor contract         : FIXED BATCH 1")
print("Held-out Test           : SEALED")
print("Progress                : 2 / 5 code cells")

PHASE 07J CELL 2 — BUNDLE AUDIT
Status                  : PASSED
Runtime contracts       : 3 / 3 VERIFIED
ONNX artifacts          : 6 / 6 VERIFIED
Vocabulary              : 10,080
Tensor contract         : FIXED BATCH 1
Held-out Test           : SEALED
Progress                : 2 / 5 code cells


## Cell 3 — V1 → V2 compatibility matrix và migration gates

In [3]:
STARTED = time.perf_counter()
V1_APP_CONTRACT = {
    "runtime": "onnxruntime-android:1.29.0",
    "assets": {
        "encoder": "baseline_encoder_fp32.onnx",
        "decoder": "baseline_decoder_fp32.onnx",
        "vocabulary": "vocabulary.json",
        "handoff": "phase7b_mobile_handoff_contract.json",
    },
    "vocabulary_size": 12293,
    "vocabulary_field": "index_to_word",
    "encoder_input_names": ["student_images"],
    "encoder_output_names": ["visual_memory"],
    "visual_memory_shape": [1, 1, 256],
    "decoder_batch_strategy": "beam_size_3_single_call",
    "decoder_batch_size": 3,
    "default_mode": "V1_FP32_DEMO",
}
V2_BUNDLE_CONTRACT = {
    "runtime": "onnxruntime-android:1.29.0",
    "assets": "three pair-specific encoder/decoder ONNX artifacts",
    "vocabulary_size": 10080,
    "vocabulary_field": "id_to_token",
    "encoder_input_names": ["image"],
    "encoder_output_names": ["visual_memory"],
    "visual_memory_shape": [1, 49, 256],
    "decoder_batch_strategy": "fixed_batch_1_sequential_beam",
    "decoder_batch_size": 1,
    "default_research_pair": "T1-S1",
    "available_pairs": ["T1-S1", "T2-S2", "T3-S3"],
}

matrix = [
    {
        "dimension": "onnxruntime_version",
        "v1": V1_APP_CONTRACT["runtime"],
        "v2": V2_BUNDLE_CONTRACT["runtime"],
        "compatible": True,
        "required_action": "retain_dependency",
    },
    {
        "dimension": "vocabulary",
        "v1": "12,293 tokens; index_to_word",
        "v2": "10,080 tokens; id_to_token",
        "compatible": False,
        "required_action": "new_v2_vocabulary_loader",
    },
    {
        "dimension": "encoder_input",
        "v1": "student_images [1,3,224,224]",
        "v2": "image [1,3,224,224]",
        "compatible": False,
        "required_action": "pair_runtime_contract_driven_input_name",
    },
    {
        "dimension": "visual_memory",
        "v1": "[1,1,256]",
        "v2": "[1,49,256]",
        "compatible": False,
        "required_action": "replace_memory_shape_and_buffer_logic",
    },
    {
        "dimension": "beam_decoder_batch",
        "v1": "batch=3 in one decoder call",
        "v2": "fixed batch=1",
        "compatible": False,
        "required_action": "run_active_beams_sequentially",
    },
    {
        "dimension": "model_selection",
        "v1": "one FP32 demo model",
        "v2": "three locked INT8 candidates",
        "compatible": False,
        "required_action": "add_explicit_pair_registry_and_selector",
    },
    {
        "dimension": "fallback",
        "v1": "existing verified demo",
        "v2": "new research runtime",
        "compatible": True,
        "required_action": "preserve_v1_as_fallback_until_device_gate",
    },
]
assert sum(not row["compatible"] for row in matrix) == 5

matrix_csv = RESULT / "phase7j_v1_v2_compatibility_matrix.csv"
with matrix_csv.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(matrix[0]))
    writer.writeheader()
    writer.writerows(matrix)

migration_requirements = {
    "schema_version": "phase7j_android_source_migration_requirements_v1",
    "created_at_utc": now(),
    "status": "migration_required",
    "direct_asset_swap_allowed": False,
    "reason": (
        "V1 and V2 differ in vocabulary, encoder input name, visual-memory "
        "shape, decoder batch contract and model-selection topology."
    ),
    "v1_contract": V1_APP_CONTRACT,
    "v2_contract": V2_BUNDLE_CONTRACT,
    "compatibility_matrix": matrix,
    "hard_requirements": [
        "Keep baseline V1 assets and Phase8CaptionEngine as fallback.",
        "Add a pair-aware V2 model registry with SHA-256 verification.",
        "Load id_to_token vocabulary with exactly 10,080 entries.",
        "Use encoder input `image` and output `visual_memory`.",
        "Allocate visual memory as [1,49,256].",
        "Execute Beam-3 through fixed-batch-1 decoder calls.",
        "Expose T1-S1, T2-S2 and T3-S3 explicitly in research UI.",
        "Record model load, preprocessing, encoder, decoder, total latency and PSS.",
        "Do not claim Android performance before physical-device benchmark.",
    ],
    "recommended_default_pair": "T1-S1",
    "default_reason": "highest Validation INT8 CIDEr-D among retained pairs",
    "physical_android_benchmark_authorized": False,
    "test_set_status": "sealed",
    "test_set_used": False,
}
write_json(
    CONFIG / "phase7j_android_source_migration_requirements.json",
    migration_requirements,
)
write_json(AUDIT / "phase7j_cell03_compatibility.json", {
    "schema_version": "phase7j_cell03_compatibility_v1",
    "created_at_utc": now(),
    "status": "passed",
    "dimensions_checked": len(matrix),
    "incompatible_dimensions": 5,
    "direct_asset_swap_allowed": False,
    "v1_fallback_required": True,
    "physical_android_benchmark_authorized": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})

print("PHASE 07J CELL 3 — COMPATIBILITY")
print("Status                  : PASSED")
print("Dimensions checked      :", len(matrix))
print("Breaking differences    : 5")
print("Direct asset swap       : FORBIDDEN")
print("V1 demo fallback        : REQUIRED")
print("Default V2 research pair: T1-S1")
print("Physical benchmark      : NOT AUTHORIZED")
print("Progress                : 3 / 5 code cells")

PHASE 07J CELL 3 — COMPATIBILITY
Status                  : PASSED
Dimensions checked      : 7
Breaking differences    : 5
Direct asset swap       : FORBIDDEN
V1 demo fallback        : REQUIRED
Default V2 research pair: T1-S1
Physical benchmark      : NOT AUTHORIZED
Progress                : 3 / 5 code cells


## Cell 4 — Build Android source-migration package

In [4]:
STARTED = time.perf_counter()
ASSETS = PACKAGE / "assets"
CONTRACTS = PACKAGE / "contracts"
KOTLIN = PACKAGE / "kotlin"
DOCS = PACKAGE / "docs"
for directory in [ASSETS, CONTRACTS, KOTLIN, DOCS]:
    directory.mkdir(parents=True, exist_ok=True)

pair_registry_rows = []
for pair, item in PAIR_RUNTIME.items():
    slug = item["registry"]["runtime_contract"]["relative_path"].split("/")[1]
    pair_asset_dir = ASSETS / slug
    pair_contract_dir = CONTRACTS / slug
    pair_asset_dir.mkdir(parents=True, exist_ok=True)
    pair_contract_dir.mkdir(parents=True, exist_ok=True)
    contract = item["contract"]

    copied_artifacts = {}
    for name, reference in item["registry"]["artifacts"].items():
        source_path = resolve_inside(BUNDLE_ROOT, reference["relative_path"])
        destination_name = (
            "encoder_int8.onnx" if name == "encoder.onnx"
            else "decoder_int8.onnx"
        )
        destination_path = pair_asset_dir / destination_name
        shutil.copy2(source_path, destination_path)
        assert sha256(destination_path) == reference["sha256"]
        copied_artifacts[name] = {
            "asset_path": destination_path.relative_to(PACKAGE).as_posix(),
            "size_bytes": destination_path.stat().st_size,
            "sha256": sha256(destination_path),
        }

    source_preprocessing = resolve_inside(
        BUNDLE_ROOT, contract["preprocessing_contract"]
    )
    preprocessing_destination = pair_contract_dir / "preprocessing.json"
    shutil.copy2(source_preprocessing, preprocessing_destination)
    runtime_destination = pair_contract_dir / "mobile_runtime_contract.json"
    shutil.copy2(item["runtime_path"], runtime_destination)

    lock_row = [
        row for row in DEPLOYMENT_LOCK["pairs"] if row["pair"] == pair
    ]
    assert len(lock_row) == 1
    pair_registry_rows.append({
        "pair": pair,
        "student_id": contract["student_id"],
        "seed": contract["seed"],
        "strategy": contract["strategy"],
        "encoder": copied_artifacts["encoder.onnx"],
        "decoder": copied_artifacts["decoder.onnx"],
        "preprocessing_contract": (
            preprocessing_destination.relative_to(PACKAGE).as_posix()
        ),
        "runtime_contract": runtime_destination.relative_to(PACKAGE).as_posix(),
        "validation": {
            "CIDEr-D": lock_row[0]["int8_CIDEr-D"],
            "BLEU-4": lock_row[0]["int8_BLEU-4"],
            "CE": lock_row[0]["int8_CE"],
            "package_reduction_fraction": (
                lock_row[0]["package_reduction_fraction"]
            ),
        },
    })

shared_destinations = {
    "v2_word_vocabulary_minfreq5.json": (
        BUNDLE_ROOT / "shared/word_vocabulary_minfreq5.json"
    ),
    "v2_tokenizer_contract.json": (
        BUNDLE_ROOT / "shared/tokenizer_contract.json"
    ),
    "v2_sequence_encoding_contract.json": (
        BUNDLE_ROOT / "shared/sequence_encoding_contract.json"
    ),
}
for destination_name, source_path in shared_destinations.items():
    shutil.copy2(source_path, ASSETS / destination_name)

registry = {
    "schema_version": "phase7j_android_model_registry_v1",
    "created_at_utc": now(),
    "runtime": "onnxruntime-android:1.29.0",
    "default_mode": "V1_FP32_DEMO",
    "default_v2_research_pair": "T1-S1",
    "v1_fallback_required": True,
    "v2_vocabulary": {
        "asset_path": "assets/v2_word_vocabulary_minfreq5.json",
        "size": 10080,
        "field": "id_to_token",
        "special_token_ids": {"pad": 0, "bos": 1, "eos": 2, "unk": 3},
    },
    "pairs": pair_registry_rows,
    "direct_asset_swap_allowed": False,
    "physical_android_benchmark_performed": False,
    "test_set_status": "sealed",
    "test_set_used": False,
}
registry_path = CONTRACTS / "phase7j_android_model_registry.json"
write_json(registry_path, registry)

kotlin_registry = """package com.klcn.mobilecaptioning

internal enum class V2ResearchPair(
    val pairId: String,
    val encoderAsset: String,
    val decoderAsset: String,
    val expectedEncoderSha256: String,
    val expectedDecoderSha256: String,
) {
    T1_S1(
        "T1-S1",
        "v2/t1_s1/encoder_int8.onnx",
        "v2/t1_s1/decoder_int8.onnx",
        "16390358c41161a54e7af0598ff87336d22bcdb3ad837f7d6917d75edcd217eb",
        "0184a94a7ae7dfc5560dfba9aebae9222337e34de4d242f2b3b5c912adcec261",
    ),
    T2_S2(
        "T2-S2",
        "v2/t2_s2/encoder_int8.onnx",
        "v2/t2_s2/decoder_int8.onnx",
        "c49e2f29b051ae2db294d4ba358de2df45104828c3058276a29839ab0e04a5c5",
        "e46e47ec3ee8087413aa81dcf2a99379f0f3485193e4a660d76d2cca62ecb7c9",
    ),
    T3_S3(
        "T3-S3",
        "v2/t3_s3/encoder_int8.onnx",
        "v2/t3_s3/decoder_int8.onnx",
        "305860fd6c25b5b8c095aa538544a9011395d46dcc653b1d72717844e183edf0",
        "72c0b16086cfb53210d35dfd48b12c8b4bd82d8223115805f41cef382f0e5304",
    ),
}

internal object Phase7JV2Contract {
    const val vocabularyAsset = "v2/v2_word_vocabulary_minfreq5.json"
    const val vocabularySize = 10_080
    const val encoderInputName = "image"
    const val encoderOutputName = "visual_memory"
    const val memoryTokens = 49
    const val hiddenSize = 256
    const val decoderSequenceLength = 31
    const val decoderBatchSize = 1
    const val beamSize = 3
}
"""
(KOTLIN / "Phase7JV2ModelRegistry.kt").write_text(
    kotlin_registry, encoding="utf-8"
)

migration_doc = """# Phase 07J Android source migration requirements

## Hard rule

Do not overwrite the V1 assets with V2 files. The contracts are incompatible.

## Required Phase 07K changes

1. Preserve `Phase8CaptionEngine` and current V1 assets as a demo fallback.
2. Copy Phase 07J assets under `app/src/main/assets/v2/`.
3. Add the generated `Phase7JV2ModelRegistry.kt`.
4. Implement a separate V2 engine driven by the model registry.
5. Parse `id_to_token` with exactly 10,080 entries.
6. Use encoder input `image` and memory `[1,49,256]`.
7. Run Beam-3 as sequential fixed-batch-1 decoder calls.
8. Add an explicit T1-S1/T2-S2/T3-S3 selector.
9. Record latency and memory without claiming physical-device results yet.
10. Build and run JVM/static contract tests before device benchmark.
"""
(DOCS / "ANDROID_SOURCE_MIGRATION_REQUIREMENTS.md").write_text(
    migration_doc, encoding="utf-8"
)

write_json(AUDIT / "phase7j_cell04_package.json", {
    "schema_version": "phase7j_cell04_package_v1",
    "created_at_utc": now(),
    "status": "passed",
    "pair_assets_packaged": 3,
    "onnx_artifacts_packaged": 6,
    "registry_generated": True,
    "kotlin_registry_generated": True,
    "v1_fallback_preserved_by_contract": True,
    "physical_android_benchmark_performed": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})

print("PHASE 07J CELL 4 — INTEGRATION PACKAGE")
print("Status                  : PASSED")
print("V2 pair assets          : 3 / 3")
print("ONNX artifacts          : 6 / 6")
print("Android registry        : GENERATED")
print("Kotlin contract         : GENERATED")
print("V1 fallback             : PRESERVED BY CONTRACT")
print("Source mutation         : NOT PERFORMED")
print("Progress                : 4 / 5 code cells")

PHASE 07J CELL 4 — INTEGRATION PACKAGE
Status                  : PASSED
V2 pair assets          : 3 / 3
ONNX artifacts          : 6 / 6
Android registry        : GENERATED
Kotlin contract         : GENERATED
V1 fallback             : PRESERVED BY CONTRACT
Source mutation         : NOT PERFORMED
Progress                : 4 / 5 code cells


## Cell 5 — Manifest, Phase 07K handoff và completion marker

In [5]:
STARTED = time.perf_counter()
manifest_rows = []
for directory in [AUDIT, CONFIG, RESULT, PACKAGE]:
    for path in sorted(directory.rglob("*")):
        if path.is_file():
            manifest_rows.append({
                "relative_path": path.relative_to(OUT).as_posix(),
                "size_bytes": path.stat().st_size,
                "sha256": sha256(path),
            })

manifest_csv = MANIFEST / "phase7j_android_integration_manifest.csv"
with manifest_csv.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(
        handle, fieldnames=["relative_path", "size_bytes", "sha256"]
    )
    writer.writeheader()
    writer.writerows(manifest_rows)
manifest_json = MANIFEST / "phase7j_android_integration_manifest.json"
write_json(manifest_json, {
    "schema_version": "phase7j_android_integration_manifest_v1",
    "created_at_utc": now(),
    "rows": manifest_rows,
})
sha_file = MANIFEST / "SHA256SUMS"
sha_file.write_text(
    "".join(
        f"{row['sha256']}  {row['relative_path']}\n"
        for row in manifest_rows
    ),
    encoding="utf-8",
)
manifest_reference = {
    "csv": {
        "relative_path": manifest_csv.relative_to(OUT).as_posix(),
        "size_bytes": manifest_csv.stat().st_size,
        "sha256": sha256(manifest_csv),
        "rows": len(manifest_rows),
    },
    "json": {
        "relative_path": manifest_json.relative_to(OUT).as_posix(),
        "size_bytes": manifest_json.stat().st_size,
        "sha256": sha256(manifest_json),
    },
    "sha256sums": {
        "relative_path": sha_file.relative_to(OUT).as_posix(),
        "size_bytes": sha_file.stat().st_size,
        "sha256": sha256(sha_file),
    },
}

migration_path = CONFIG / "phase7j_android_source_migration_requirements.json"
registry_path = CONTRACTS / "phase7j_android_model_registry.json"
next_ready = {
    "schema_version": "phase7k_android_source_migration_input_ready_v1",
    "created_at_utc": now(),
    "status": "ready",
    "phase": "android_source_migration_and_build_preflight",
    "migration_requirements": {
        "relative_path": migration_path.relative_to(OUT).as_posix(),
        "size_bytes": migration_path.stat().st_size,
        "sha256": sha256(migration_path),
    },
    "android_model_registry": {
        "relative_path": registry_path.relative_to(OUT).as_posix(),
        "size_bytes": registry_path.stat().st_size,
        "sha256": sha256(registry_path),
    },
    "integration_package_root": "android_integration_package",
    "v1_fallback_required": True,
    "direct_asset_swap_allowed": False,
    "android_source_migration_authorized": True,
    "physical_android_benchmark_authorized": False,
    "new_training": False,
    "checkpoint_reselection": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "manifest": manifest_reference,
}
write_json(
    OUT / "PHASE7K_ANDROID_SOURCE_MIGRATION_INPUT_READY.json",
    next_ready,
)

marker = {
    "schema_version": (
        "phase7j_android_integration_compatibility_preflight_complete_v1"
    ),
    "created_at_utc": now(),
    "status": "complete",
    "decision": "source_migration_required_phase7k_authorized",
    "pairs_verified": 3,
    "onnx_artifacts_verified": 6,
    "direct_asset_swap_allowed": False,
    "v1_fallback_required": True,
    "integration_package_complete": True,
    "android_source_mutated": False,
    "physical_android_benchmark_performed": False,
    "new_training_performed": False,
    "checkpoint_reselection_performed": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "manifest": manifest_reference,
    "manifest_rows": len(manifest_rows),
    "elapsed_seconds_cell05": time.perf_counter() - STARTED,
}
write_json(
    OUT / "PHASE7J_ANDROID_INTEGRATION_COMPATIBILITY_PREFLIGHT_COMPLETE.json",
    marker,
)

print("PHASE 07J — FINAL")
print("Status                  : COMPLETE")
print("Pair bundles            : 3 / 3 VERIFIED")
print("Direct asset swap       : FORBIDDEN")
print("Migration package       : COMPLETE")
print("Android source mutation : NOT PERFORMED")
print("Phase 07K               : AUTHORIZED")
print("Physical benchmark      : NOT PERFORMED")
print("Held-out Test           : SEALED / NOT USED")
print("Manifest                :", len(manifest_rows), "/", len(manifest_rows), "VERIFIED")
print("Progress                : 5 / 5 code cells")

PHASE 07J — FINAL
Status                  : COMPLETE
Pair bundles            : 3 / 3 VERIFIED
Direct asset swap       : FORBIDDEN
Migration package       : COMPLETE
Android source mutation : NOT PERFORMED
Phase 07K               : AUTHORIZED
Physical benchmark      : NOT PERFORMED
Held-out Test           : SEALED / NOT USED
Manifest                : 25 / 25 VERIFIED
Progress                : 5 / 5 code cells
